In [12]:
countries = [
    "United States",
    "Saudi Arabia",
    "China",
    "Germany",
    "Japan",
    "United Kingdom",
    "Cayman Islands",
    "Brazil",
    "Taiwan",
    "Switzerland",
    "Netherlands",
    "South Korea",
    "France",
    "India",
    "Norway",
    "Italy",
    "Thailand",
    "Canada",
    "Spain",
    "Poland",
    "Ireland",
    "Turkey",
    "Hong Kong",
    "Luxembourg",
    "Belgium",
    "Denmark",
    "Australia",
    "Mexico",
    "Sweden",
    "Singapore",
    "Portugal",
    "South Africa",
    "Austria",
    "Colombia",
    "Chile",
    "Hungary",
    "Argentina",
    "Finland",
    "Bermuda",
    "Indonesia",
    "Israel",
    "Malaysia",
    "Czech Republic",
    "New Zealand",
    "United Arab Emirates",
    "Greece",
    "Qatar",
    "Vietnam",
    "Philippines",
    "Kuwait",
    "Isle of Man",
    "Kazakhstan",
    "Jersey",
    "Bahrain",
    "Uruguay",
    "Macau",
    "Panama",
    "Cyprus",
    "Romania",
    "Guernsey",
    "Egypt",
    "Peru",
    "Iceland",
    "Monaco",
    "Bahamas",
    "Nigeria",
    "Estonia",
    "Liechtenstein",
    "Jordan",
    "Lithuania",
    "Gibraltar",
    "Costa Rica",
    "Cambodia",
    "Malta",
    "Oman",
    "Pakistan",
    "Sudan",
    "Russia",
    "British Virgin Islands"
]

sectors = [
    'Energy',
    'Materials',
    'Industrials',
    'Consumer Discretionary',
    'Consumer Staples',
    'Health Care',
    'Financials',
    'Information Technology',
    'Communication Services',
    'Utilities',
    'Real Estate'
]

import pandas as pd

# Ler os dois CSVs (como texto para evitar erros de tipo)
done_df = pd.read_csv('enriched_ns_batches_done.csv', dtype=str)
unique_df = pd.read_csv('unique_org_info_pipeline_multi_ns_dominant_as_enriched.csv', dtype=str)

# Garantir que as colunas estão na mesma ordem e formato
common_cols = list(set(done_df.columns) & set(unique_df.columns))

# Concatenar mantendo apenas as colunas compatíveis
combined_df = pd.concat(
    [unique_df[common_cols], done_df[common_cols]],
    ignore_index=True
).drop_duplicates()

# Ordenar as colunas na mesma ordem do arquivo base (unique_df)
combined_df = combined_df[unique_df.columns]

#transform revenue column to numeric for sorting

combined_df['revenue'] = pd.to_numeric(combined_df['revenue'], errors='coerce')

#sort by revenue in descending order
combined_df = combined_df.sort_values(by='revenue', ascending=False)

#exclude lines that "country_name" is empty or not in countries list

combined_df = combined_df[combined_df['country_name'].isin(countries)]

#exclude lines where sector is empty or not in the list of sectors
combined_df = combined_df[combined_df['sector'].isin(sectors)]



# Salvar o resultado final
combined_df.head(10000).to_csv('unique_org_info_pipeline_multi_ns_enriched_combined.csv', index=False)

#list unique sectors in the final file
unique_sectors = combined_df['sector'].unique()
print("unique sectors = ", unique_sectors)


unique sectors =  ['Consumer Staples' 'Consumer Discretionary' 'Energy' 'Health Care'
 'Information Technology' 'Financials' 'Communication Services'
 'Industrials' 'Materials' 'Utilities' 'Real Estate']


In [ ]:
import pandas as pd
import numpy as np
import re

# Carregue seu CSV padrao domain,as_number,NS,NS-A,company_name,revenue,country_name,sector,rank,prefix,prefix_length,aut_name,org_id,opaque_id,source_x,org_name,country,source_y
df = pd.read_csv("./unique_org_info_pipeline_multi_ns_dominant_as_enriched.csv")

# --- Normalizações leves ---
# revenue como float
df["revenue"] = pd.to_numeric(df["revenue"], errors="coerce").fillna(0.0)

# provider_country: é a coluna 'country' (código de país do ASN); company_country: 'country_name'
df = df.rename(columns={"country": "provider_country", "country_name": "company_country"})


# --- Blocos econômicos (apenas flags) ---
EU = {
    "Germany","France","Netherlands","Italy","Spain","Poland","Ireland","Luxembourg","Belgium",
    "Denmark","Sweden","Portugal","Austria","Hungary","Czech Republic","Finland","Greece",
    "Cyprus","Estonia","Lithuania","Malta","Romania"
}
G20 = {
    "United States","Saudi Arabia","China","Germany","Japan","United Kingdom","Brazil","France","India",
    "Italy","Canada","Australia","Mexico","South Korea","Turkey","Indonesia","South Africa","Argentina"
}
BRICS = {"Brazil","Russia","India","China","South Africa","Saudi Arabia"}  # status 2025

def which_block(country):
    if country in EU:
        return "EU"
    if country in G20:
        return "G20"
    if country in BRICS:
        return "BRICS"
    return "Other"

df["block"] = df["company_country"].apply(which_block)

# (Opcional) onshore flag
df["is_onshore"] = (df["company_country"] == df["provider_country"])

display(df.head())

,domain,as_number,NS,NS-A,company_name,revenue,company_country,sector,rank,prefix,prefix_length,aut_name,org_id,opaque_id,source_x,org_name,provider_country,source_y,block,is_onshore
0,walmart.com,21342,a5-65.akam.net,95.100.168.65,Walmart,6.931530e+11,United States,Consumer Staples,1.0,95.100.168.0,24.0,AKAMAI-ASN2,ORG-AT1-RIPE,NaN,RIPE,Akamai International B.V.,NL,RIPE,G20,False
1,amazon.com,12008,ns2.amzndns.net,156.154.69.10,Amazon,6.700380e+11,United States,Consumer Discretionary,2.0,156.154.69.0,24.0,SECURITYSERVICES,SSL-1134-ARIN,d46edd86fd49dba22360500c6b8657bf_ARIN,ARIN,"Vercara, LLC",US,ARIN,G20,False
2,aramco.com,21342,a7-67.akam.net,23.61.199.67,Saudi Aramco,4.615681e+11,Saudi Arabia,Energy,3.0,23.61.199.0,24.0,AKAMAI-ASN2,ORG-AT1-RIPE,NaN,RIPE,Akamai International B.V.,NL,RIPE,G20,False
3,sinopec.com,9808,ns6.sinopec.com,111.56.240.132,Sinopec,4.447496e+11,China,Energy,4.0,111.56.240.0,21.0,CHINAMOBILE-CN,ORG-CM1-AP-APNIC,A91D5F3D_APNIC,APNIC,China Mobile,CN,APNIC,G20,False
4,unitedhealthgroup.com,10879,epla1.corpnamesvcs.com,149.111.128.204,UnitedHealth,4.228180e+11,United States,Health Care,5.0,149.111.128.0,19.0,UHC,UNITED-215-ARIN,d39768e24c8472eb3b02486a17098b02_ARIN,ARIN,Unitedhealthcare,US,ARIN,G20,False


In [5]:
def hhi(group, weight_col="revenue", key="dns_provider"):
    w = group.groupby(key, dropna=False)[weight_col].sum()
    if w.sum() == 0:
        return np.nan
    s = w / w.sum()
    return float((s**2).sum())

def crk(group, k=3, weight_col="revenue", key="dns_provider"):
    w = group.groupby(key, dropna=False)[weight_col].sum().sort_values(ascending=False)
    if w.sum() == 0:
        return np.nan
    return float(w.head(k).sum() / w.sum())

def rar_by_provider_country(group, weight_col="revenue"):
    return (
        group.groupby("provider_country")[weight_col]
             .sum()
             .sort_values(ascending=False)
             .reset_index(name="RAR")
    )

def domestic_bias(group, weight_col="revenue"):
    tot = group[weight_col].sum()
    if tot == 0:
        return np.nan
    return float(group.loc[group["is_onshore"], weight_col].sum() / tot)

def neff(group, weight_col="revenue", key="dns_provider"):
    # número efetivo de provedores = 1/HHI
    h = hhi(group, weight_col, key)
    return (1.0 / h) if pd.notna(h) and h > 0 else np.nan

In [6]:
agg_list = []
for (b, s), g in df.groupby(["block", "sector"]):
    agg_list.append({
        "block": b,
        "sector": s,
        "HHI_provider": hhi(g, "revenue", "dns_provider"),
        "CR3_provider": crk(g, k=3, weight_col="revenue", key="dns_provider"),
        "CR5_provider": crk(g, k=5, weight_col="revenue", key="dns_provider"),
        "N_eff_provider": neff(g, "revenue", "dns_provider"),
        "Revenue_total": g["revenue"].sum(),
        "Companies": g["company_name"].nunique(),
        "Providers": g["dns_provider"].nunique()
    })
tbl_conc = pd.DataFrame(agg_list).sort_values(["block","sector"]).reset_index(drop=True)
# tbl_conc -> para Fig.1 (HHI), Fig.2 (CR3/CR5), Fig.8 (scatter HHI x Revenue)

KeyError: 'dns_provider'